# CSIS3754 Exam Q3 – 2025
## Spider Measurements — Unsupervised Learning

## 3.1 — Import dataset with named columns

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

spiders = pd.read_csv('spider_measurements.csv', header=None,
                      names=['abdomen-circumference','abdomen-length',
                             'pedipalp-length','chelicerae-length'])
print(spiders.head())

## 3.2 — Brief summary

In [ ]:
# Statistical summary
print(spiders.describe())

# Concise summary
spiders.info()

# Missing values
print('Missing values:')
print(spiders.isnull().sum())

## 3.3 — Pre-processing

In [ ]:
print('Pre-processing steps:')
print('1. No object columns — all features are numeric measurements, no encoding needed.')
print('2. Checking for missing values...')
print(spiders.isnull().sum())
print('3. Scaling features using StandardScaler so all measurements contribute equally to clustering.')

scaler = StandardScaler()
spiders_scaled = scaler.fit_transform(spiders)

print('\nNo object columns remaining:')
print(spiders.select_dtypes(include=['object']).columns.tolist())
print('Shape after pre-processing:', spiders_scaled.shape)

## 3.4 — k-Means clustering

In [ ]:
# Elbow curve
inertia = []
k_range = range(2, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(spiders_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o', color='blue')
plt.title('Elbow Curve')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

In [ ]:
# Silhouette scores
sil_scores = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(spiders_scaled)
    sil_scores.append(silhouette_score(spiders_scaled, labels))

plt.figure(figsize=(8, 5))
plt.plot(k_range, sil_scores, marker='o', color='orange')
plt.title('Silhouette Score per k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

best_k = list(k_range)[sil_scores.index(max(sil_scores))]
print(f'Best k by silhouette score: {best_k} (score = {max(sil_scores):.4f})')

In [ ]:
# Train final model
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
spiders['Cluster'] = kmeans.fit_predict(spiders_scaled)

print('Cluster labels:')
print(spiders['Cluster'].values)
print('\nCluster counts:')
print(spiders['Cluster'].value_counts())

## 3.5 — PCA to 2 dimensions + scatter plot

In [ ]:
# Apply PCA
pca = PCA(n_components=2)
spiders_pca = pca.fit_transform(spiders_scaled)

print('Original dataset dimensions: ', spiders_scaled.shape)
print('PCA-transformed dimensions:  ', spiders_pca.shape)

print('\nPrincipal components (loadings):')
print(pca.components_)

print('\nExplained variance ratio:')
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {var:.4f} ({var*100:.2f}%)')
print(f'  Total: {pca.explained_variance_ratio_.sum():.4f} ({pca.explained_variance_ratio_.sum()*100:.2f}%)')

In [ ]:
# Scatter plot of PCA components with clusters
pca_df = pd.DataFrame(spiders_pca, columns=['PC1', 'PC2'])
pca_df['Cluster'] = spiders['Cluster']

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Cluster',
                palette='tab10', s=60, alpha=0.8)
plt.title('PCA Scatter Plot — Spider Clusters')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.tight_layout()
plt.show()

## 3.6 — Evaluation and discussion

In [ ]:
print(f"""
=== Cluster Evaluation ===

Number of clusters identified: {best_k}

Scatter Plot Evaluation:
The PCA scatter plot reduces the 4-dimensional spider measurement data
to 2 dimensions, capturing {pca.explained_variance_ratio_.sum()*100:.1f}% of the total variance.
If the clusters are visibly separated in the scatter plot, this confirms
that the k-means algorithm correctly identified natural groupings in
the spider body part measurements.

PCA Effectiveness:
PCA was effective in reducing the dimensions from 4 to 2 while retaining
a significant proportion of the original variance. This allows the clusters
to be visualised in a 2D scatter plot, which would not have been possible
with the original 4-dimensional data.

Agreement with k value:
If the scatter plot shows {best_k} clearly separated groups, the chosen
k={best_k} is appropriate. If the groups overlap significantly, a different
k value or additional features may be needed for better separation.
""")